# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayesha-Shahzadkhan/flyrank-assignment1/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
!git clone https://github.com/Ayesha-Shahzadkhan/flyrank-assignment1.git
%cd flyrank-assignment1


Cloning into 'flyrank-assignment1'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 163 (delta 70), reused 99 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.88 MiB | 14.58 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/flyrank-assignment1/flyrank-assignment1


In [23]:
import os
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [24]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()

(30000, 44)


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


Method: Logistic Regression

Why: The Week-4 baseline was a simple rule — if impressions were high,
predict decline. Logistic Regression does something similar but learns
the weights instead of using a fixed cutoff, and uses more than one
feature at a time. It also stays easy to explain, since each feature
gets a clear weight — useful for a decision-support tool where I need
to say *why* something was flagged.

In [25]:
from sklearn.linear_model import LogisticRegression

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [26]:
df['is_declining'] = (df['trend_direction']=='down').astype(int)

In [27]:
from sklearn.model_selection import train_test_split

leak_cols = [
    'trend_direction', 'trend_pct',
    'impressions_last_30d', 'impressions_prev_30d',
    'clicks_last_30d', 'clicks_prev_30d',
    'sessions_last_30d', 'sessions_prev_30d',
]

X = df.drop(columns=leak_cols + ['is_declining']).select_dtypes(include='number')
X = X.fillna(X.median())
y = df['is_declining']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [28]:
print(x_test.columns.tolist())

['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


In [29]:
baseline_pred = (x_test['impressions_90d']>81).astype(int)
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

model_pred = model.predict(x_test)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = pd.DataFrame({
    'method': ['Baseline', 'Logistic Regression'],
    'accuracy': [accuracy_score(y_test, baseline_pred), accuracy_score(y_test,model_pred)],
    'precision': [precision_score(y_test, baseline_pred), precision_score(y_test, model_pred)],
    'recall': [recall_score(y_test, baseline_pred), precision_score(y_test, model_pred)],
    'f1': [f1_score(y_test, baseline_pred), f1_score(y_test, model_pred)],

})
results

,method,accuracy,precision,recall,f1
0,Baseline,0.599000,0.594042,0.821648,0.689548
1,Logistic Regression,0.643667,0.656813,0.656813,0.685773


In [31]:
import numpy as np
coefs = pd.Series(model.coef_[0], index=x_train.columns).sort_values(key=np.abs, ascending=False)
print(coefs.head(10))

days_with_sessions       -0.027719
days_with_impressions     0.018744
scroll_events_90d         0.009955
avg_position             -0.008200
ctr                      -0.007104
engagement_rate          -0.004931
users_90d                -0.004539
sessions_90d              0.003242
days_since_last_update    0.003174
content_age_days         -0.003108
dtype: float64


Logistic Regression edges out the baseline on accuracy and precision,
but the baseline still recalls more actual decline cases (0.822 vs
0.657). F1 scores are nearly identical, so this isn't a clear win —
it's a trade-off: the model is more precise but misses more true
declines than the simple rule does.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [32]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, model_pred)
print(cm)

[[1529 1219]
 [ 919 2333]]


Confusion matrix: 919 false negatives (missed declines) vs 1219 false
positives (false alarms). The model errs more often toward flagging
content that wasn't actually declining, rather than missing real
declines — a safer failure mode for a decision-support tool, since
over-flagging costs review time but under-flagging lets real decline
go unaddressed.

Top features the model leans on: days_with_impressions,
days_with_sessions, pageviews_90d, age_tier_order — mostly volume/
recency-of-activity signals rather than a single dominant feature,
unlike the baseline's single-threshold rule.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.